# Fair by which definition?

MichAl Academy, lesson 2.10.

Run each cell with **Shift+Enter**.

Every model in this track has been judged by one number at a time. This notebook
judges one model by four at once, and finds that improving any of them makes at
least one of the others worse.

That is not a flaw in the model. It is arithmetic, proved in 2016 and 2017, and
the point of running it is to see the shape of the trade before somebody asks
you to "just make it fair".

**A note on the data.** This is the only notebook in the track that downloads
anything. No dataset bundled with scikit-learn carries a protected attribute,
and a lesson about fairness needs a real one, so this fetches the Adult census
extract from OpenML. It is about 4 MB and Colab and Kaggle both allow it.

**A note on the subject.** These are real 1994 US census records about real
people's incomes, and the disparities below are historical facts about that
population, not artefacts. The model did not create them. What the model does is
decide what to do about them, which is the part you are responsible for.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 0
raw = fetch_openml("adult", version=2, as_frame=True, parser="auto")

# fnlwgt is a census sampling weight, not a fact about the person. Leaving it in
# is a small version of the leakage from lesson 2.3.
df = raw.data.drop(columns=["fnlwgt"])
y = (raw.target == ">50K").astype(int).to_numpy()
group = df["sex"].astype(str).to_numpy()

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    df, y, group, test_size=0.3, random_state=SEED, stratify=y
)
print(f"{len(df):,} records, {df.shape[1]} columns, {len(y_test):,} held back")
print(f"the task: predict whether this person earns more than 50,000 dollars")


## 1. Look at the base rates first

Before fitting anything. The impossibility result below only applies when the
groups have different base rates, so this cell decides whether the rest of the
notebook is relevant.


In [ ]:
print(f"overall: {y.mean():.4f} earn over 50K")
for g in ("Male", "Female"):
    m = group == g
    print(f"  {g:<7} n={m.sum():6,}   base rate {y[m].mean():.4f}")
ratio = y[group == "Male"].mean() / y[group == "Female"].mean()
print(f"\nratio: {ratio:.2f} to 1")


0.3038 against 0.1093, a ratio of 2.78 to 1.

Read that carefully, because it is the single most consequential number in this
notebook and it says nothing about any model. It is a fact about the 1994 US
labour market, and every difficulty below follows from it.


## 2. Fit one model and judge it four ways

The four criteria you will be asked about, in plain terms:

- **Demographic parity.** The same share of each group gets selected.
- **Equal opportunity.** Of the people who really do earn over 50K, the same
  share gets found in each group. That is the true positive rate.
- **Predictive equality.** Of the people who do not, the same share gets wrongly
  flagged in each group. The false positive rate.
- **Predictive parity.** When the model says yes, it is right equally often in
  each group. Precision.


In [ ]:
cat_cols = [c for c in df.columns if str(df[c].dtype) in ("category", "object")]
num_cols = [c for c in df.columns if c not in cat_cols]


def make_model(columns):
    cats = [c for c in columns if c in cat_cols]
    nums = [c for c in columns if c in num_cols]
    return Pipeline([
        ("pre", ColumnTransformer([
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20,
                                  sparse_output=False), cats),
            ("num", StandardScaler(), nums),
        ])),
        ("clf", HistGradientBoostingClassifier(random_state=SEED)),
    ])


model = make_model(list(df.columns)).fit(X_train, y_train)
score = model.predict_proba(X_test)[:, 1]
print(f"accuracy {((score >= 0.5).astype(int) == y_test).mean():.4f}"
      f"   ROC-AUC {roc_auc_score(y_test, score):.4f}")


def rates(mask, threshold, s=None):
    """The four rates for one group at one threshold, counted not looked up."""
    s = score if s is None else s
    yy, ss = y_test[mask], s[mask]
    a = ss >= threshold
    tp = int((a & (yy == 1)).sum()); fp = int((a & (yy == 0)).sum())
    fn = int((~a & (yy == 1)).sum()); tn = int((~a & (yy == 0)).sum())
    return {
        "selected": (tp + fp) / len(yy),
        "tpr": tp / (tp + fn) if tp + fn else np.nan,
        "fpr": fp / (fp + tn) if fp + tn else np.nan,
        "precision": tp / (tp + fp) if tp + fp else np.nan,
    }


def compare(t_male, t_female, label, s=None):
    sc = score if s is None else s
    rm = rates(g_test == "Male", t_male, s)
    rf = rates(g_test == "Female", t_female, s)
    flag = np.where(g_test == "Male", sc >= t_male, sc >= t_female)
    acc = (flag.astype(int) == y_test).mean()
    print(f"\n{label}   (thresholds: men {t_male:.2f}, women {t_female:.2f})"
          f"   accuracy {acc:.4f}")
    print(f"  {'':<8} {'selected':>9} {'TPR':>8} {'FPR':>8} {'precision':>10}")
    for nm, r in (("men", rm), ("women", rf)):
        print(f"  {nm:<8} {r['selected']:>9.4f} {r['tpr']:>8.4f}"
              f" {r['fpr']:>8.4f} {r['precision']:>10.4f}")
    print(f"  {'gap':<8} {rm['selected'] - rf['selected']:>+9.4f}"
          f" {rm['tpr'] - rf['tpr']:>+8.4f} {rm['fpr'] - rf['fpr']:>+8.4f}"
          f" {rm['precision'] - rf['precision']:>+10.4f}")
    return rm, rf


rm_base, rf_base = compare(0.5, 0.5, "A. one threshold for everyone")


One threshold, four verdicts:

**Demographic parity: badly failed.** 25.4% of men selected against 8.0% of
women, a gap of 17 points.

**Equal opportunity: failed.** The model finds 65.8% of the high earners among
men and 56.4% among women.

**Predictive equality: failed.** 7.8% of men who do not earn over 50K get
flagged anyway, against 2.0% of women. Four times the rate.

**Predictive parity: passed, almost exactly.** Precision 0.7868 and 0.7832, a
gap of 0.0037.

So "is this model fair?" has no answer. It depends entirely on which of the four
you meant, and this model already satisfies one of them to within four
thousandths.


## 3. The fix everyone tries first, and why it fails

The intuitive move is to stop telling the model about sex. If it never sees the
column, it cannot discriminate on it.

Test it.


In [ ]:
blind_cols = [c for c in df.columns if c != "sex"]
blind = make_model(blind_cols).fit(X_train[blind_cols], y_train)
score_blind = blind.predict_proba(X_test[blind_cols])[:, 1]

print(f"{'model':<16} {'accuracy':>9} {'sel men':>9} {'sel women':>10} {'TPR gap':>9}")
for label, sc in (("with sex", score), ("sex removed", score_blind)):
    rm = rates(g_test == "Male", 0.5, sc)
    rf = rates(g_test == "Female", 0.5, sc)
    acc = ((sc >= 0.5).astype(int) == y_test).mean()
    print(f"{label:<16} {acc:>9.4f} {rm['selected']:>9.4f}"
          f" {rf['selected']:>10.4f} {rm['tpr'] - rf['tpr']:>+9.4f}")


Accuracy 0.8729 to 0.8728. Selection rates and the true positive rate gap
essentially unmoved.

Removing the column did nothing at all, and the next cell shows why.


In [ ]:
# If the model can rebuild the column from the others, deleting it is theatre.
sex_from_rest = make_model(blind_cols).fit(X_train[blind_cols],
                                           (g_train == "Male").astype(int))
p = sex_from_rest.predict_proba(X_test[blind_cols])[:, 1]
truth = (g_test == "Male").astype(int)
print(f"predicting sex from the other 12 columns:")
print(f"  accuracy {((p >= 0.5).astype(int) == truth).mean():.4f}"
      f"   ROC-AUC {roc_auc_score(truth, p):.4f}")
print(f"  always guessing 'Male' would score {truth.mean():.4f}")


ROC-AUC 0.9361. The column is almost perfectly recoverable from `relationship`,
`occupation`, `hours-per-week` and `marital-status`, so the model reconstructs
it whether you supply it or not.

This has a name, **fairness through unawareness**, and it is the one approach in
this notebook that is simply wrong rather than a trade-off. Worse, it removes
your ability to measure the disparity while leaving the disparity in place.

Which is the first practical rule: **you cannot audit what you do not collect.**
Deleting a protected attribute from the training features is a decision worth
arguing about. Deleting it from the evaluation set is not.


## 4. Now try to fix one criterion at a time

Different thresholds per group is the bluntest available tool. It is also the
clearest way to see the trade, so use it here and read section 6 before using it
anywhere real.

Pick a female threshold that matches the male true positive rate, then the false
positive rate, then the selection rate. Watch the fourth column.


In [ ]:
GRID = np.round(np.arange(0.02, 0.99, 0.01), 2)


def match_female(key, target):
    """The female threshold whose `key` lands closest to the male value."""
    best, gap = GRID[0], np.inf
    for t in GRID:
        v = rates(g_test == "Female", t)[key]
        if not np.isnan(v) and abs(v - target) < gap:
            best, gap = t, abs(v - target)
    return best


compare(0.5, match_female("tpr", rm_base["tpr"]),
        "B. equalise the true positive rate")
compare(0.5, match_female("fpr", rm_base["fpr"]),
        "C. equalise the false positive rate")
compare(0.5, match_female("selected", rm_base["selected"]),
        "D. equalise the selection rate")


Read the precision column down the four blocks. It is the same model throughout.

```
A. one threshold          precision gap  +0.0037
B. equal TPR                             +0.0841
C. equal FPR                             +0.2229
D. equal selection rate                  +0.3974
```

Closing the selection-rate gap opens a precision gap of 0.40. Of the women the
model flags, 39% really earn over 50K; of the men, 79%. Two people with the same
prediction from the same model now have completely different chances of that
prediction being right, and which one you are depends on your sex.

That is not a bug and it is not fixable by better engineering. Chouldechova put
it plainly in 2017: "the criteria cannot all be simultaneously satisfied when
recidivism prevalence differs across groups." Her subject was recidivism; the
arithmetic does not care. Kleinberg, Mullainathan and Raghavan proved the same
incompatibility independently in 2016.


## 5. So is there any setting that satisfies all four?

Search every threshold pair, 97 by 97, and take the one that minimises the
largest of the four gaps.


In [ ]:
best = None
for tm in GRID:
    rm = rates(g_test == "Male", tm)
    for tf in GRID:
        rf = rates(g_test == "Female", tf)
        if np.isnan(rf["precision"]) or np.isnan(rm["precision"]):
            continue
        worst = max(abs(rm[k] - rf[k]) for k in ("selected", "tpr", "fpr", "precision"))
        if best is None or worst < best[0]:
            best = (worst, tm, tf, rm, rf)

worst, tm, tf, rm, rf = best
print(f"best pair: men {tm:.2f}, women {tf:.2f}   largest remaining gap {worst:.4f}")
compare(tm, tf, "E. minimise the largest gap")

flagged = np.zeros(len(y_test), dtype=bool)
flagged[g_test == "Male"] = score[g_test == "Male"] >= tm
flagged[g_test == "Female"] = score[g_test == "Female"] >= tf
print(f"\nshare of everyone flagged: {flagged.mean():.4f}")
print(f"accuracy here {(flagged.astype(int) == y_test).mean():.4f}"
      f"   against {((score >= 0.5).astype(int) == y_test).mean():.4f} at one threshold")


There is a setting. It puts both thresholds near the top of the scale, flags
4.9% of everyone, finds about a fifth of the real high earners, and costs six
points of accuracy, 0.8729 down to 0.8093.

The theorem allows exactly two escapes: equal base rates, or a perfect
classifier. What the search found is the practical shadow of the second one. Make
the model select almost nobody and there is not enough left for the criteria to
disagree about.


## 6. What this does and does not authorise

Two cautions that belong beside the arithmetic.

**Per-group thresholds are used here as a measuring instrument.** Setting a
different bar for men and women is disparate treatment, and in hiring, lending
and housing it is illegal in many jurisdictions. This notebook uses it because
it is the cleanest way to see the trade, not because it is a deployment
strategy. Real work goes into the features, the labels and the data collection.

**Which criterion is correct is not a technical question.** Barocas, Hardt and
Narayanan's textbook is the place to go next, and its position is that the
criteria encode different moral commitments about what the decision is for.
Nothing you can compute will tell you which commitment your organisation has
made. What the measurements above tell you is the price of each one, which is
the part a model can contribute to that conversation.


## What to take from this

| Claim | What we measured |
|---|---|
| This model is fair | Meaningless alone. It passes predictive parity to 0.0037 and fails the other three |
| Removing the protected attribute removes the bias | No. Accuracy and the TPR gap barely moved, and sex is recoverable from the rest at ROC-AUC 0.9361 |
| A fairer threshold costs a little accuracy | Not the whole cost. Equalising selection rates opened a precision gap of 0.40, which no accuracy figure shows you |
| The criteria can all be satisfied with enough care | No. Proved impossible at unequal base rates, and here they differ 2.78 to 1 |
| There is always a fair setting available | Only a near-trivial one: 4% flagged, a fifth of the real cases found, six points of accuracy gone |

The habit to carry: **before arguing about whether a model is fair, write down
which of the four you mean and measure the base rates.** If the base rates
differ, you are choosing between the criteria whether you admit it or not.


## Try this

1. Use `race` as the protected attribute instead of `sex`. There are more than
   two groups, so the criteria have to hold pairwise. Does anything get easier?
2. Equalise the precision gap deliberately, by matching on `precision` instead.
   Which of the other three moves furthest?
3. Calibrate the model per group with lesson 2.8's `CalibratedClassifierCV`,
   fitted separately on each group. Calibration within groups is itself one of
   the fairness criteria in the literature. Check whether making both groups
   honestly calibrated changes any of the four gaps at a single threshold.
